# Arquitectura OLAP en Spotify (Dataset Real 2024) 🎵
## Contexto:
Eres el Arquitecto de Datos Principal en una disquera. El equipo de ingeniería extrajo un dump transaccional masivo con las canciones más reproducidas de Spotify en 2024. El archivo plano es ancho, redundante y poco óptimo para análisis.

Tu misión es aplicar una tubería ETL para convertir este archivo crudo en un Esquema de Estrella en memoria, y posteriormente responder a 10 preguntas críticas del equipo de analítica utilizando el Cubo OLAP.

### Fase 1: Extract (Lectura del archivo crudo)

Leemos el CSV tal como viene (formato OLTP: una sola tabla ancha y redundante).

In [1]:
import pandas as pd
import numpy as np

# EXTRACT: Leemos directamente el archivo crudo desde el repositorio
url_dataset = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"

# Para evitar errores de codificación con caracteres especiales en nombres de artistas
# thousands=',' convierte "390,470,936" en el número 390470936
spotify_raw = pd.read_csv(url_dataset, encoding='latin-1', thousands=',')

print("Muestra de datos crudos (OLTP):")
display(spotify_raw.head(3))
print("Columnas disponibles:", spotify_raw.columns.tolist())

Muestra de datos crudos (OLTP):


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,390470936.0,30716.0,196631588.0,...,684.0,62.0,17598718.0,114.0,18004655.0,22931.0,4818457.0,2669262.0,NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,323703884.0,28113.0,174597137.0,...,3.0,67.0,10422430.0,111.0,7780028.0,28444.0,6623075.0,1118279.0,NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,601309283.0,54331.0,211607669.0,...,536.0,136.0,36321847.0,172.0,5022621.0,5639.0,7208651.0,5285340.0,NaN,0


Columnas disponibles: ['Track', 'Album Name', 'Artist', 'Release Date', 'ISRC', 'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts', 'TIDAL Popularity', 'Explicit Track']


### Fase 2: Transform & Load (Construyendo el Almacén)

Observa las columnas del dataset. Tienes texto (Track, Artist, Date) mezclado con métricas puras (Spotify Streams, YouTube Views). 

**Reto Arquitectónico:**
1. Crea una **Dimensión Tiempo (`dim_tiempo`)**: Extrae las columnas relacionadas al año, mes y día de lanzamiento. Elimina duplicados y crea un `id_tiempo`.
2. Crea una **Dimensión Pista (`dim_pista`)**: Extrae el nombre del track, el artista y si es explícita. Crea un `id_pista`.
3. Crea la **Tabla de Hechos (`fact_streams`)**: Cruza los IDs y conserva **únicamente** métricas numéricas (ej. Spotify Streams, YouTube Views, Track Score) junto con las llaves foráneas. NO debe haber texto aquí.

In [2]:
# --- ESCRIBE TU PIPELINE ETL AQUÍ ---

# 1. Crear dim_tiempo
dim_tiempo = spotify_raw[["Release Date"]].drop_duplicates().reset_index(drop=True)
fecha = pd.to_datetime(dim_tiempo["Release Date"])
dim_tiempo["Year"] = fecha.dt.year
dim_tiempo["Month"] = fecha.dt.month
dim_tiempo["Day"] = fecha.dt.day
dim_tiempo["id_tiempo"] = dim_tiempo.index +1
print(dim_tiempo.head(3))

  Release Date  Year  Month  Day  id_tiempo
0    4/26/2024  2024      4   26          1
1     5/4/2024  2024      5    4          2
2    3/19/2024  2024      3   19          3


In [3]:
# 2. Crear dim_pista
dim_pista = spotify_raw[["Track","Artist","Explicit Track"]].drop_duplicates().reset_index(drop=True)
dim_pista["id_pista"] = dim_pista.index +1
print(dim_pista.head(3))

                        Track          Artist  Explicit Track  id_pista
0         MILLION DOLLAR BABY   Tommy Richman               0         1
1                 Not Like Us  Kendrick Lamar               1         2
2  i like the way you kiss me         Artemas               0         3


In [4]:
# 3. Crear fact_streams
# Pegamos los IDs de las dimensiones a cada fila del archivo crudo
fact_streams = spotify_raw.merge(dim_tiempo, on="Release Date", how="left").merge(dim_pista, on=["Track","Artist","Explicit Track"], how="left")

# Nos quedamos SOLO con las llaves foráneas y las métricas numéricas (nada de texto)
fact_streams = fact_streams[["id_tiempo","id_pista","Track Score","Spotify Streams","YouTube Views","TikTok Views","Spotify Popularity"]]
print(fact_streams.head(3))


   id_tiempo  id_pista  Track Score  Spotify Streams  YouTube Views  \
0          1         1        725.4      390470936.0     84274754.0   
1          2         2        545.9      323703884.0    116347040.0   
2          3         3        538.4      601309283.0    122599116.0   

   TikTok Views  Spotify Popularity  
0  5.332282e+09                92.0  
1  2.083390e+08                92.0  
2  3.369121e+09                92.0  


In [5]:
# 4. (LOAD) Reconstruir el Cubo en memoria para las consultas
# Unimos la tabla de hechos con sus dimensiones usando las llaves
cubo_olap = fact_streams.merge(dim_tiempo, on="id_tiempo", how="left").merge(dim_pista, on="id_pista", how="left")
cubo_olap


,id_tiempo,id_pista,Track Score,Spotify Streams,YouTube Views,TikTok Views,Spotify Popularity,Release Date,Year,Month,Day,Track,Artist,Explicit Track
0,1,1,725.4,3.904709e+08,8.427475e+07,5.332282e+09,92.0,4/26/2024,2024,4,26,MILLION DOLLAR BABY,Tommy Richman,0
1,2,2,545.9,3.237039e+08,1.163470e+08,2.083390e+08,92.0,5/4/2024,2024,5,4,Not Like Us,Kendrick Lamar,1
2,3,3,538.4,6.013093e+08,1.225991e+08,3.369121e+09,92.0,3/19/2024,2024,3,19,i like the way you kiss me,Artemas,0
3,4,4,444.9,2.031281e+09,1.096101e+09,1.460373e+10,85.0,1/12/2023,2023,1,12,Flowers,Miley Cyrus,0
4,5,5,423.3,1.070349e+08,7.737396e+07,NaN,88.0,5/31/2024,2024,5,31,Houdini,Eminem,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4595,1497,4532,19.4,3.050500e+08,1.492477e+08,1.439344e+08,71.0,9/5/2017,2017,9,5,For the Last Time,$uicideboy$,1
4596,1030,4533,19.4,5.228236e+07,9.439202e+08,1.993500e+04,56.0,7/27/2018,2018,7,27,Dil Meri Na Sune,Atif Aslam,0
4597,746,4534,19.4,1.899727e+08,2.010273e+08,3.684900e+07,65.0,2/28/2020,2020,2,28,Grace (feat. 42 Dugg),Lil Baby,1
4598,1561,4535,19.4,1.454670e+08,1.118595e+09,3.775730e+07,66.0,11/8/2016,2016,11,8,Nashe Si Chadh Gayi,Arijit Singh,0


### Explica porqué solo hay 2 tablas dimensiones o si podrías incluir alguna otra.

Solo hay 2 dimensiones porque el ejercicio pide las dos preguntas más básicas que se le hacen a una canción: **¿cuándo salió?** (`dim_tiempo`) y **¿qué canción es y de quién?** (`dim_pista`). Con esas dos ya podemos contestar los 10 retos.

Pero sí se podrían agregar más dimensiones, porque el archivo crudo todavía tiene texto descriptivo que no usamos:

- **`dim_album`**: con la columna `Album Name`. Así podríamos agrupar streams por álbum.
- **`dim_artista`**: separar `Artist` de la pista. Un artista tiene muchas canciones, así que no repetiríamos su nombre en cada fila (esto se acerca a un esquema copo de nieve).
- **`dim_plataforma`**: las métricas vienen de Spotify, YouTube, TikTok, Apple Music, Deezer, Amazon, Pandora, etc. Si "volteamos" la tabla para tener una fila por (canción, plataforma), la plataforma se vuelve una dimensión y podríamos comparar plataformas fácilmente.
- El **`ISRC`** (código único internacional de la grabación) podría ir en `dim_pista` como identificador natural.

Columnas como `All Time Rank` o `Track Score` NO son dimensiones: son números que se pueden sumar o promediar, por eso van en la tabla de hechos.

**¿Y una dimensión de popularidad?** Pensé en agregar una tercera dimensión de *Popularity*, pero no tenía mucho sentido: `Spotify Popularity` (al igual que `Track Score`, `Spotify Streams` o `YouTube Views`) es un número que se puede promediar o sumar, o sea, es una **métrica**, no una dimensión. Y justamente de eso se trata la tabla de hechos: de llenarla con métricas, mientras que las dimensiones solo describen el *quién*, el *qué* y el *cuándo* (texto o categorías). Por eso la popularidad se queda en `fact_streams`.


### Fase 3: Consultas Analíticas (10 Retos)

El equipo de analítica te ha enviado 10 peticiones. Para cada una, debes:
1. Escribir el código en Pandas usando la matriz `cubo_olap` que construiste.
2. Identificar y escribir en un comentario qué tipo de operación OLAP estás aplicando: **Roll-up, Drill-down, Slice o Dice**.

In [6]:
# RETO 1: El CEO quiere ver el total acumulado de 'Spotify Streams' agrupado por 'Artist'.
# Operación OLAP: Roll-up (pasamos de nivel canción a nivel artista: sumamos y vemos menos detalle)
respuesta_1 = cubo_olap.groupby("Artist")["Spotify Streams"].sum().sort_values(ascending=False)
display(respuesta_1)

# RETO 2: El equipo de marketing quiere ver todas las métricas únicamente de las canciones lanzadas en el año 2024.
# Operación OLAP: Slice (fijamos UN valor en UNA dimensión: Year = 2024)
respuesta_2 = cubo_olap[cubo_olap["Year"] == 2024]
display(respuesta_2)

# RETO 3: Finanzas requiere el total de 'Spotify Streams' desglosado primero por Año de Lanzamiento (Year) y luego detallado por Mes de Lanzamiento (Month).
# Operación OLAP: Drill-down (bajamos de Año a Año + Mes: más detalle)
respuesta_3 = cubo_olap.groupby(["Year", "Month"])["Spotify Streams"].sum()
display(respuesta_3)

# RETO 4: Auditoría te pide extraer el rendimiento de las canciones que son explícitas Y que además fueron lanzadas en el año 2024.
# Operación OLAP: Dice (filtramos por DOS dimensiones a la vez: pista explícita y año 2024)
respuesta_4 = cubo_olap[(cubo_olap["Explicit Track"] == 1) & (cubo_olap["Year"] == 2024)]
display(respuesta_4)

# RETO 5: Queremos disminuir el nivel de detalle y ver solamente el promedio del 'Track Score' agrupado por Año de Lanzamiento.
# Operación OLAP: Roll-up (resumimos todas las canciones a nivel año)
respuesta_5 = cubo_olap.groupby("Year")["Track Score"].mean()
display(respuesta_5)


Artist
Bad Bunny        3.705483e+10
The Weeknd       3.694854e+10
Drake            3.496216e+10
Taylor Swift     3.447077e+10
Post Malone      2.613747e+10
                     ...     
3 Palavrinhas    0.000000e+00
Leo Walrus       0.000000e+00
Los Del Rio      0.000000e+00
Lori             0.000000e+00
Lo Greco Bros    0.000000e+00
Name: Spotify Streams, Length: 1999, dtype: float64

,id_tiempo,id_pista,Track Score,Spotify Streams,YouTube Views,TikTok Views,Spotify Popularity,Release Date,Year,Month,Day,Track,Artist,Explicit Track
0,1,1,725.4,390470936.0,84274754.0,5.332282e+09,92.0,4/26/2024,2024,4,26,MILLION DOLLAR BABY,Tommy Richman,0
1,2,2,545.9,323703884.0,116347040.0,2.083390e+08,92.0,5/4/2024,2024,5,4,Not Like Us,Kendrick Lamar,1
2,3,3,538.4,601309283.0,122599116.0,3.369121e+09,92.0,3/19/2024,2024,3,19,i like the way you kiss me,Artemas,0
4,5,5,423.3,107034922.0,77373957.0,NaN,88.0,5/31/2024,2024,5,31,Houdini,Eminem,1
6,7,7,407.2,900158751.0,308723145.0,5.349153e+08,86.0,1/18/2024,2024,1,18,Beautiful Things,Benson Boone,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4544,361,4481,19.6,6184259.0,1090122.0,NaN,67.0,5/16/2024,2024,5,16,ATM,Nemzzz,1
4552,13,4489,19.6,3970523.0,408201.0,NaN,67.0,5/17/2024,2024,5,17,AirBnb,ian,1
4561,190,4498,19.5,1633647.0,9700826.0,NaN,NaN,5/24/2024,2024,5,24,"Tu Juliet Jatt Di ( From ""Jatt & Juliet 3"")",Diljit Dosanjh,0
4593,616,4530,19.4,3754914.0,34315313.0,NaN,NaN,3/27/2024,2024,3,27,"Jaragandi (From ""Game Changer"") (Telugu)",Thaman S,0


Year  Month
1987  7        1.879386e+09
1991  9        2.021910e+09
1994  1        1.810650e+09
1998  4        1.783399e+08
1999  6        1.405354e+09
                   ...     
2024  2        1.023077e+10
      3        1.263673e+10
      4        2.152035e+10
      5        1.005348e+10
      6        2.711085e+09
Name: Spotify Streams, Length: 191, dtype: float64

,id_tiempo,id_pista,Track Score,Spotify Streams,YouTube Views,TikTok Views,Spotify Popularity,Release Date,Year,Month,Day,Track,Artist,Explicit Track
1,2,2,545.9,3.237039e+08,116347040.0,2.083390e+08,92.0,5/4/2024,2024,5,4,Not Like Us,Kendrick Lamar,1
4,5,5,423.3,1.070349e+08,77373957.0,NaN,88.0,5/31/2024,2024,5,31,Houdini,Eminem,1
7,8,8,375.8,6.750792e+08,228382568.0,3.804584e+09,92.0,2/2/2024,2024,2,2,Gata Only,FloyyMenor,1
8,9,9,355.7,1.653018e+09,NaN,NaN,NaN,6/9/2024,2024,6,9,Danza Kuduro - Cover,MUSIC LAB JPN,1
9,10,10,330.6,9.067657e+07,32735244.0,9.746562e+08,86.0,5/23/2024,2024,5,23,BAND4BAND (feat. Lil Baby),Central Cee,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4486,123,4425,19.7,1.594834e+07,2127850.0,NaN,NaN,4/5/2024,2024,4,5,7 Minute Drill,J. Cole,1
4491,132,4430,19.7,9.126485e+06,2292547.0,NaN,NaN,5/3/2024,2024,5,3,Never Last,Lil Tecca,1
4544,361,4481,19.6,6.184259e+06,1090122.0,NaN,67.0,5/16/2024,2024,5,16,ATM,Nemzzz,1
4552,13,4489,19.6,3.970523e+06,408201.0,NaN,67.0,5/17/2024,2024,5,17,AirBnb,ian,1


Year
1987    21.500000
1991    19.600000
1994    31.500000
1998    20.800000
1999    22.200000
2000    30.400000
2001    20.100000
2002    26.060000
2003    23.766667
2004    24.240000
2005    27.140000
2006    22.300000
2007    23.860000
2008    23.854545
2009    26.000000
2010    31.425926
2011    31.800000
2012    32.197872
2013    31.674545
2014    31.678571
2015    37.988542
2016    37.828455
2017    37.949765
2018    36.977510
2019    39.380405
2020    38.042222
2021    38.144118
2022    41.693228
2023    43.264594
2024    54.120058
Name: Track Score, dtype: float64

In [7]:
# RETO 6: Muestra el comportamiento analítico aislando únicamente los datos del artista "Bad Bunny".
# Operación OLAP: Slice (un solo valor en la dimensión pista: Artist = "Bad Bunny")
respuesta_6 = cubo_olap[cubo_olap["Artist"] == "Bad Bunny"]
display(respuesta_6)

# RETO 7: Aumenta el nivel de detalle: Muestra el total de 'YouTube Views' agrupado por Artista, y dentro de cada Artista, desglosado por Nombre de la Canción (Track).
# Operación OLAP: Drill-down (de Artista bajamos a Artista + Canción)
respuesta_7 = cubo_olap.groupby(["Artist", "Track"])["YouTube Views"].sum()
display(respuesta_7)

# RETO 8: Extrae un sub-cubo que contenga únicamente canciones lanzadas en el Mes 12 (Diciembre) del Año 2023.
# Operación OLAP: Dice (dos condiciones al mismo tiempo: Year = 2023 y Month = 12, sale un sub-cubo)
respuesta_8 = cubo_olap[(cubo_olap["Year"] == 2023) & (cubo_olap["Month"] == 12)]
display(respuesta_8)

# RETO 9: Resume la información calculando el total histórico de 'Spotify Streams' de todos los tiempos, agrupado por Mes (para saber en qué mes históricamente se lanza la música más exitosa).
# Operación OLAP: Roll-up (juntamos todos los años y solo dejamos el Mes)
respuesta_9 = cubo_olap.groupby("Month")["Spotify Streams"].sum().sort_values(ascending=False)
display(respuesta_9)

# RETO 10: Extrae un bloque de información filtrando las canciones de "Taylor Swift" que NO sean explícitas y hayan salido en 2024.
# Operación OLAP: Dice (tres condiciones en varias dimensiones: artista, explícita y año)
respuesta_10 = cubo_olap[(cubo_olap["Artist"] == "Taylor Swift") & (cubo_olap["Explicit Track"] == 0) & (cubo_olap["Year"] == 2024)]
display(respuesta_10)


,id_tiempo,id_pista,Track Score,Spotify Streams,YouTube Views,TikTok Views,Spotify Popularity,Release Date,Year,Month,Day,Track,Artist,Explicit Track
63,57,61,184.8,1.811991e+09,1.516863e+09,6.470937e+09,79.0,5/6/2022,2022,5,6,Me Porto Bonito,Bad Bunny,1
88,57,86,163.1,1.514240e+09,1.366181e+09,5.249864e+09,77.0,5/6/2022,2022,5,6,Titï¿½ï¿½ Me Pregu,Bad Bunny,1
102,91,100,150.7,4.781509e+08,1.133797e+08,2.111289e+09,78.0,10/13/2023,2023,10,13,MONACO,Bad Bunny,1
174,91,171,114.2,6.891815e+08,1.833815e+08,4.874404e+08,83.0,10/13/2023,2023,10,13,PERRO NEGRO,Bad Bunny,1
187,57,184,110.7,1.421763e+09,9.458571e+08,2.251428e+08,78.0,5/6/2022,2022,5,6,Ojitos Lindos,Bad Bunny,0
245,204,242,98.4,7.178490e+08,2.176877e+08,8.085188e+08,72.0,5/18/2023,2023,5,18,WHERE SHE GOES,Bad Bunny,1
261,135,258,96.1,1.977947e+09,1.861597e+09,2.157336e+07,72.0,1/1/2020,2020,1,1,Dakiti,Bad Bunny,1
271,57,268,94.1,1.108032e+09,7.657802e+08,9.406307e+08,77.0,5/6/2022,2022,5,6,Moscow Mule,Bad Bunny,1
290,57,287,90.1,1.293774e+09,5.019082e+08,6.046490e+09,77.0,5/6/2022,2022,5,6,Efecto,Bad Bunny,1
294,238,291,88.6,1.398634e+09,1.084872e+09,2.877719e+09,71.0,6/4/2021,2021,6,4,Yonaguni,Bad Bunny,1


Artist                                       Track                                                              
"XY"                                         Come and Say - Original Mix                                            415159148.0
$OHO BANI                                    ZEIT, DASS SICH WAS DREHT                                                6744201.0
$uicideboy$                                  ...And to Those I Love, Thanks for Sticking Around                     269460083.0
                                             1000 Blunts                                                             87680985.0
                                             Are You Going to See the Rose in the Vase, or the Dust on the Table      2640169.0
                                                                                                                       ...     
ýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýý              ýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýý                              44

,id_tiempo,id_pista,Track Score,Spotify Streams,YouTube Views,TikTok Views,Spotify Popularity,Release Date,Year,Month,Day,Track,Artist,Explicit Track
61,55,59,186.5,1384.0,222076.0,8.811417e+09,NaN,12/31/2023,2023,12,31,Very Cute Melody by Marimba Tone (39813),mitsu sound,0
94,84,92,155.9,649404552.0,433839212.0,1.060356e+07,83.0,12/2/2023,2023,12,2,La Diabla,Xavi,0
97,87,95,153.1,359803580.0,369321334.0,8.740562e+08,86.0,12/7/2023,2023,12,7,BELLAKEO,Peso Pluma,1
131,112,129,130.7,8327.0,NaN,7.331825e+07,NaN,12/10/2023,2023,12,10,Looking Like a Wow End of Year 2023,WasteLand,0
177,152,174,112.6,606124553.0,166363445.0,2.960124e+07,86.0,12/1/2023,2023,12,1,LUNA,Feid,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4189,979,4133,20.7,100634965.0,19367095.0,NaN,NaN,12/22/2023,2023,12,22,Tacata (Remix),Tiagz,1
4194,87,4138,20.7,9129729.0,78290967.0,NaN,NaN,12/7/2023,2023,12,7,YA NO TE EXTRAï¿½,NATTI NATASHA,0
4385,152,4325,20.1,45238166.0,8478416.0,1.026006e+07,NaN,12/1/2023,2023,12,1,MY HOUSE,Beyoncï¿,1
4462,1533,4401,19.8,3225791.0,72510719.0,NaN,NaN,12/29/2023,2023,12,29,ýýýýýýýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýýýýýýýýýýý,0


Month
1     3.652168e+11
5     1.779293e+11
11    1.734614e+11
10    1.672667e+11
3     1.638041e+11
7     1.503832e+11
6     1.484293e+11
4     1.462846e+11
8     1.426826e+11
9     1.363626e+11
2     1.333565e+11
12    1.022498e+11
Name: Spotify Streams, dtype: float64

,id_tiempo,id_pista,Track Score,Spotify Streams,YouTube Views,TikTok Views,Spotify Popularity,Release Date,Year,Month,Day,Track,Artist,Explicit Track
16,17,17,297.6,395433400.0,104673836.0,207317000.0,89.0,4/18/2024,2024,4,18,Fortnight (feat. Post Malone),Taylor Swift,0
866,17,861,52.3,166168092.0,16423009.0,7202700.0,78.0,4/18/2024,2024,4,18,Whoï¿½ï¿½ï¿½s Afraid of Little O,Taylor Swift,0
1013,17,1007,47.8,175613480.0,15237337.0,1510800.0,84.0,4/18/2024,2024,4,18,"So Long, London",Taylor Swift,0
1167,17,1159,44.0,161916445.0,13509432.0,NaN,83.0,4/18/2024,2024,4,18,My Boy Only Breaks His Favorite Toys,Taylor Swift,0
1270,17,1260,42.1,165611453.0,9873283.0,6383500.0,78.0,4/18/2024,2024,4,18,Guilty as Sin?,Taylor Swift,0
2484,183,2454,28.6,97601615.0,5528058.0,17893741.0,87.0,4/19/2024,2024,4,19,So High School,Taylor Swift,0
2517,17,2486,28.3,106201248.0,5521347.0,NaN,NaN,4/18/2024,2024,4,18,The Alchemy,Taylor Swift,0
2528,17,2497,28.2,250430303.0,NaN,NaN,NaN,4/18/2024,2024,4,18,I Can Do It With a Broken Heart,Taylor Swift,0
2776,183,2744,26.7,88244010.0,7381186.0,14644000.0,NaN,4/19/2024,2024,4,19,imgonnagetyouback,Taylor Swift,0
2816,17,2784,26.5,105424914.0,7227379.0,2500000.0,NaN,4/18/2024,2024,4,18,Fresh Out The Slammer,Taylor Swift,0
